In [1]:
# ==========================================
# 1. INSTALL DEPENDENCIES & SETUP
# ==========================================
import sys
import subprocess

print("Installing dependencies... (This may take a moment)")
subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/elastic/ember.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])

import os
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef
import time
import warnings
import ember

warnings.filterwarnings('ignore')

# ==========================================
# 2. CONFIGURATION
# ==========================================
class Config:
    # Directory for EMBER data
    DATA_DIR = './ember2018'

    # EMBER 2018 (Version 2) has 2381 features
    INPUT_DIM = 2381
    LATENT_DIM = 64
    N_WAY = 2  # Benign vs Malicious

    # Autoencoder Params
    AE_LR = 0.001
    AE_EPOCHS = 10  # Reduced for demo speed (increase for better results)
    BATCH_SIZE = 256 # Larger batch size for EMBER

    # Meta-Learning Params
    META_LR = 0.005
    INNER_LR = 0.01
    INNER_STEPS = 5
    META_EPISODES = 500
    TEST_EPISODES = 50
    Q_QUERY = 50 # Reduced query set per episode to manage RAM

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 3. DATA DOWNLOAD & PREPARATION
# ==========================================
def prepare_ember_data():
    if not os.path.exists(Config.DATA_DIR):
        os.makedirs(Config.DATA_DIR)

    # check if vectorized data exists
    if not os.path.exists(os.path.join(Config.DATA_DIR, "X_train.dat")):
        print(f"Downloading EMBER 2018 dataset to {Config.DATA_DIR}...")
        # Download
        tar_file = "ember_dataset_2018_2.tar.bz2"
        if not os.path.exists(tar_file):
            subprocess.check_call(["wget", "https://ember.elastic.co/ember_dataset_2018_2.tar.bz2"])

        print("Extracting dataset...")
        subprocess.check_call(["tar", "-xvf", tar_file])

        # Move extracted files to DATA_DIR if they aren't already there
        # The tar usually extracts to a folder named 'ember2018'
        if os.path.exists("ember2018") and os.path.abspath("ember2018") != os.path.abspath(Config.DATA_DIR):
             # Move contents if paths differ
             pass

        print("Vectorizing features (This is RAM intensive and takes time)...")
        ember.create_vectorized_features(Config.DATA_DIR)
        print("Vectorization complete.")
    else:
        print("EMBER dataset already found and vectorized.")

# ==========================================
# 4. CUSTOM DATASET LOADER
# ==========================================
class EmberRamDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_data_loaders():
    print("Loading vectorized EMBER features into memory...")
    # EMBER helper to read .dat files
    # X_train is (900000, 2381), y_train is (900000,)
    X_train, y_train, X_test, y_test = ember.read_vectorized_features(Config.DATA_DIR)

    # Filter out unlabeled data (label = -1)
    train_rows = (y_train != -1)
    X_train = X_train[train_rows]
    y_train = y_train[train_rows]

    # For Few-Shot Sampling, we need to separate classes
    # 0 = Benign, 1 = Malicious
    print("Separating Malicious and Benign samples...")

    # We will use a subset to save RAM if necessary, e.g., first 100k of each
    # Adjust slicing [:] if you run out of RAM
    X_benign = X_train[y_train == 0]
    X_malicious = X_train[y_train == 1]

    # Create a standard DataLoader for Autoencoder Training (Unsupervised/Self-supervised style)
    # We use all available training data
    dataset = EmberRamDataset(X_train, y_train)
    train_loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

    print(f"Data Loaded. Benign: {len(X_benign)}, Malicious: {len(X_malicious)}")

    # Convert to tensors for the MAML sampler
    X_ben_tensor = torch.tensor(X_benign, dtype=torch.float32)
    X_mal_tensor = torch.tensor(X_malicious, dtype=torch.float32)

    return X_mal_tensor, X_ben_tensor, train_loader

# ==========================================
# 5. MODEL DEFINITIONS
# ==========================================
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 1024), nn.ReLU(),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 512), nn.ReLU(),
            nn.Linear(512, 1024), nn.ReLU(),
            nn.Linear(1024, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 6. HELPER FUNCTIONS
# ==========================================
def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    # Sample Support
    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    # Sample Query
    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)

    q_size = min(Config.Q_QUERY, len(remaining_mal), len(remaining_ben))

    idx_mal_q = np.random.choice(remaining_mal, q_size, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, q_size, replace=False)

    # Batches
    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]])
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long()

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]])
    y_qry = torch.cat([torch.ones(q_size), torch.zeros(q_size)]).long()

    # Extract Features using Autoencoder
    with torch.no_grad():
        z_sup, _ = ae_model(X_sup.to(Config.DEVICE))
        z_qry, _ = ae_model(X_qry.to(Config.DEVICE))

    return z_sup, y_sup.to(Config.DEVICE), z_qry, y_qry.to(Config.DEVICE)

def manual_maml_step(model, z_sup, y_sup, z_qry, y_qry, train=True):
    fast_weights = dict(model.named_parameters())

    # Inner Loop
    for _ in range(Config.INNER_STEPS):
        out = torch.func.functional_call(model, fast_weights, z_sup)
        loss = nn.CrossEntropyLoss()(out, y_sup)
        grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=train)
        fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

    # Outer Loop
    out_q = torch.func.functional_call(model, fast_weights, z_qry)

    if train:
        meta_loss = nn.CrossEntropyLoss()(out_q, y_qry)
        return meta_loss
    else:
        preds = torch.argmax(out_q, dim=1)
        return preds, y_qry

def calculate_metrics(y_true, y_pred):
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()
    return (accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average='binary'),
            precision_score(y_true, y_pred, average='binary'),
            recall_score(y_true, y_pred, average='binary'),
            matthews_corrcoef(y_true, y_pred))

# ==========================================
# 7. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":

    # 1. Prepare Data
    prepare_ember_data()
    X_mal, X_ben, train_loader = get_data_loaders()

    # 2. Train Autoencoder
    print(f"\n[Stage 1] Training Autoencoder for {Config.AE_EPOCHS} epochs on EMBER features...")
    ae = Autoencoder().to(Config.DEVICE)
    opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)

    ae.train()
    for epoch in range(Config.AE_EPOCHS):
        total_loss = 0
        batch_count = 0
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(Config.DEVICE)

            # Forward
            _, recon = ae(batch_x)
            loss = nn.MSELoss()(recon, batch_x)

            # Backward
            opt_ae.zero_grad()
            loss.backward()
            opt_ae.step()

            total_loss += loss.item()
            batch_count += 1

        print(f"Epoch {epoch+1}/{Config.AE_EPOCHS} - Loss: {total_loss/batch_count:.6f}")

    ae.eval()
    print("Autoencoder training complete.\n")

    # 3. MAML Evaluation
    k_values = [1, 5, 20] # Reduced list for demonstration speed
    results_summary = []

    print(f"Starting Sequential Evaluation for K = {k_values}")

    for k in k_values:
        print(f"\nRunning for K={k}-Shot")

        maml = MAMLClassifier().to(Config.DEVICE)
        meta_optimizer = optim.Adam(maml.parameters(), lr=Config.META_LR)
        scheduler = optim.lr_scheduler.StepLR(meta_optimizer, step_size=200, gamma=0.5)

        # Meta-Training
        maml.train()
        start_time = time.time()
        for episode in range(Config.META_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae, k)

            meta_optimizer.zero_grad()
            loss = manual_maml_step(maml, z_s, y_s, z_q, y_q, train=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(maml.parameters(), 1.0)
            meta_optimizer.step()
            scheduler.step()

            if (episode+1) % 100 == 0:
                print(f"Episode {episode+1}/{Config.META_EPISODES}, Loss: {loss.item():.4f}")

        print(f"   Training finished in {time.time()-start_time:.1f}s")

        # Meta-Testing
        metrics_accum = {"Acc": [], "F1": [], "Prec": [], "Rec": [], "MCC": []}
        for _ in range(Config.TEST_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae, k)
            preds, targets = manual_maml_step(maml, z_s, y_s, z_q, y_q, train=False)

            acc, f1, prec, rec, mcc = calculate_metrics(targets, preds)
            metrics_accum["Acc"].append(acc)
            metrics_accum["F1"].append(f1)
            metrics_accum["Prec"].append(prec)
            metrics_accum["Rec"].append(rec)
            metrics_accum["MCC"].append(mcc)

        res = {
            "K-Shot": k,
            "Accuracy": np.mean(metrics_accum["Acc"]),
            "F1-Score": np.mean(metrics_accum["F1"]),
            "Precision": np.mean(metrics_accum["Prec"]),
            "Recall": np.mean(metrics_accum["Rec"]),
            "MCC": np.mean(metrics_accum["MCC"])
        }
        results_summary.append(res)
        print(f"   Result: Acc={res['Accuracy']:.4f}, F1={res['F1-Score']:.4f}")

    # 4. Final Output
    print(f"\n{'K-Shot':<10} | {'Accuracy':<12} | {'F1-Score':<12} | {'Precision':<12} | {'Recall':<12} | {'MCC':<12}")
    for res in results_summary:
        print(f"{res['K-Shot']:<10} | {res['Accuracy']:.4f}       | {res['F1-Score']:.4f}       | {res['Precision']:.4f}       | {res['Recall']:.4f}       | {res['MCC']:.4f}")

Installing dependencies... (This may take a moment)


ModuleNotFoundError: No module named 'lief'

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
import time


class Config:
    DATASET_PATH = '/content/Dataset.csv'

    INPUT_DIM = 72
    LATENT_DIM = 64
    N_WAY = 2


    K_SHOT_COMPARISON = 5
    Q_QUERY = 100


    AE_LR = 0.001
    AE_EPOCHS = 50
    BATCH_SIZE = 64

    META_LR = 0.005
    INNER_LR = 0.01
    INNER_STEPS = 5
    META_EPISODES = 500
    TEST_EPISODES = 50

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def preprocess_dataset(filepath):
    print(f"Loading dataset from {filepath}...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print("Error: Dataset file not found.")
        return None, None

    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    target_col = None
    for col in ['Class', 'class', 'label']:
        if col in df.columns: target_col = col; break

    if target_col:
        y = df[target_col]
        X = df.drop(columns=cols_to_drop + [target_col])
    else:
        y = df.iloc[:, -1]
        X = df.iloc[:, :-1]

    # Binary Encoding: Benign=0, Malware=1
    y_enc = y.apply(lambda x: 0 if str(x).lower().strip() == 'benign' else 1).values

    # Feature Processing
    X_processed = X.copy()
    for col in X_processed.columns:
        if X_processed[col].dtype == 'object':
            try:
                X_processed[col] = X_processed[col].apply(
                    lambda x: int(str(x), 16) if str(x).startswith('0x') else float(x)
                )
            except:
                le = LabelEncoder()
                X_processed[col] = le.fit_transform(X_processed[col].astype(str))

    scaler = MinMaxScaler()
    X_final = scaler.fit_transform(X_processed)

    # Padding/Truncating to 72
    if X_final.shape[1] < 72:
        pad = np.zeros((X_final.shape[0], 72 - X_final.shape[1]))
        X_final = np.hstack([X_final, pad])
    elif X_final.shape[1] > 72:
        X_final = X_final[:, :72]

    return torch.FloatTensor(X_final), torch.LongTensor(y_enc)


# Base Standard Autoencoder
class StandardAE(nn.Module):
    def __init__(self):
        super(StandardAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return z, recon

# Variational Autoencoder (VAE)
class VariationalAE(nn.Module):
    def __init__(self):
        super(VariationalAE, self).__init__()
        # Encoder parts
        self.fc1 = nn.Linear(Config.INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, Config.LATENT_DIM)
        self.fc_logvar = nn.Linear(128, Config.LATENT_DIM)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return z, recon, mu, logvar

# MAML Classifier
class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.weights = nn.ParameterDict({
            'w1': nn.Parameter(torch.randn(Config.LATENT_DIM, 128) * 0.01),
            'b1': nn.Parameter(torch.zeros(128)),
            'w2': nn.Parameter(torch.randn(128, 64) * 0.01),
            'b2': nn.Parameter(torch.zeros(64)),
            'w3': nn.Parameter(torch.randn(64, Config.N_WAY) * 0.01),
            'b3': nn.Parameter(torch.zeros(Config.N_WAY))
        })

    def forward(self, x, params=None):
        if params is None: params = self.weights
        x = F.linear(x, params['w1'].t(), params['b1']); x = F.relu(x)
        x = F.linear(x, params['w2'].t(), params['b2']); x = F.relu(x)
        x = F.linear(x, params['w3'].t(), params['b3'])
        return x


def train_autoencoder(model_type, model, train_loader):
    optimizer = optim.Adam(model.parameters(), lr=Config.AE_LR)
    model.train()

    print(f"--> Training {model_type}...")
    for epoch in range(Config.AE_EPOCHS):
        total_loss = 0
        for (batch_x,) in train_loader:
            batch_x = batch_x.to(Config.DEVICE)
            optimizer.zero_grad()

            # --- Specific Training Logic ---
            if model_type == 'Denoising AE':
                # Add noise to input, reconstruct original
                noise = torch.randn_like(batch_x) * 0.1
                noisy_x = batch_x + noise
                z, recon = model(noisy_x)
                loss = nn.MSELoss()(recon, batch_x)

            elif model_type == 'Variational AE':
                # VAE Loss = MSE + KL Divergence
                z, recon, mu, logvar = model(batch_x)
                mse_loss = nn.MSELoss(reduction='sum')(recon, batch_x)
                # KL Divergence
                kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                loss = (mse_loss + kld_loss * 0.001) / batch_x.size(0) # Weight KLD small

            elif model_type == 'Sparse AE':
                # Add L1 Penalty on Z
                z, recon = model(batch_x)
                mse_loss = nn.MSELoss()(recon, batch_x)
                l1_penalty = torch.mean(torch.abs(z))
                loss = mse_loss + (0.0001 * l1_penalty)

            else: # Standard AE
                z, recon = model(batch_x)
                loss = nn.MSELoss()(recon, batch_x)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    return model


def evaluate_feature_extractor(name, ae_model, X_mal, X_ben):
    maml = MAMLClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(maml.parameters(), lr=Config.META_LR)

    # Meta-Training
    maml.train()
    ae_model.eval()

    with torch.no_grad():
        if name == 'Variational AE':
            # VAE returns z, recon, mu, logvar. We want mu.
            _, _, mu_mal, _ = ae_model(X_mal.to(Config.DEVICE))
            _, _, mu_ben, _ = ae_model(X_ben.to(Config.DEVICE))
            feats_mal = mu_mal.cpu()
            feats_ben = mu_ben.cpu()
        else:
            z_mal, _ = ae_model(X_mal.to(Config.DEVICE))
            z_ben, _ = ae_model(X_ben.to(Config.DEVICE))
            feats_mal = z_mal.cpu()
            feats_ben = z_ben.cpu()

    # MAML Loop
    for _ in range(Config.META_EPISODES):
        # Sample
        s_mal_idx = np.random.choice(len(feats_mal), Config.K_SHOT_COMPARISON)
        s_ben_idx = np.random.choice(len(feats_ben), Config.K_SHOT_COMPARISON)
        q_mal_idx = np.random.choice(len(feats_mal), Config.Q_QUERY)
        q_ben_idx = np.random.choice(len(feats_ben), Config.Q_QUERY)

        s_x = torch.cat([feats_mal[s_mal_idx], feats_ben[s_ben_idx]]).to(Config.DEVICE)
        s_y = torch.cat([torch.ones(Config.K_SHOT_COMPARISON), torch.zeros(Config.K_SHOT_COMPARISON)]).long().to(Config.DEVICE)
        q_x = torch.cat([feats_mal[q_mal_idx], feats_ben[q_ben_idx]]).to(Config.DEVICE)
        q_y = torch.cat([torch.ones(Config.Q_QUERY), torch.zeros(Config.Q_QUERY)]).long().to(Config.DEVICE)

        # Inner Loop
        fast_weights = {n: p.clone() for n, p in maml.weights.items()}
        for _ in range(Config.INNER_STEPS):
            out = maml(s_x, params=fast_weights)
            loss = F.cross_entropy(out, s_y)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        # Outer Loop
        out_q = maml(q_x, params=fast_weights)
        meta_loss = F.cross_entropy(out_q, q_y)

        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    # --- Meta-Testing ---
    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        s_mal_idx = np.random.choice(len(feats_mal), Config.K_SHOT_COMPARISON)
        s_ben_idx = np.random.choice(len(feats_ben), Config.K_SHOT_COMPARISON)
        q_mal_idx = np.random.choice(len(feats_mal), Config.Q_QUERY)
        q_ben_idx = np.random.choice(len(feats_ben), Config.Q_QUERY)

        s_x = torch.cat([feats_mal[s_mal_idx], feats_ben[s_ben_idx]]).to(Config.DEVICE)
        s_y = torch.cat([torch.ones(Config.K_SHOT_COMPARISON), torch.zeros(Config.K_SHOT_COMPARISON)]).long().to(Config.DEVICE)
        q_x = torch.cat([feats_mal[q_mal_idx], feats_ben[q_ben_idx]]).to(Config.DEVICE)
        q_y_cpu = torch.cat([torch.ones(Config.Q_QUERY), torch.zeros(Config.Q_QUERY)]).long()

        # Adaptation
        fast_weights = {n: p.clone() for n, p in maml.weights.items()}
        for _ in range(Config.INNER_STEPS):
            out = maml(s_x, params=fast_weights)
            loss = F.cross_entropy(out, s_y)
            grads = torch.autograd.grad(loss, fast_weights.values())
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        with torch.no_grad():
            out_q = maml(q_x.to(Config.DEVICE), params=fast_weights)
            preds = torch.argmax(out_q, dim=1).cpu()

        accs.append(accuracy_score(q_y_cpu, preds))
        f1s.append(f1_score(q_y_cpu, preds))

    return np.mean(accs), np.mean(f1s)


if __name__ == "__main__":
    X, y = preprocess_dataset(Config.DATASET_PATH)

    if X is not None:
        dataset = TensorDataset(X)
        loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

        # Split Data for Evaluation
        X_mal = X[y == 1]
        X_ben = X[y == 0]

        models_to_test = {
            "Standard AE": StandardAE().to(Config.DEVICE),
            "Denoising AE": StandardAE().to(Config.DEVICE),
            "Sparse AE": StandardAE().to(Config.DEVICE),
            "Variational AE": VariationalAE().to(Config.DEVICE)
        }

        results = []

        print(f"\nStarting Comparison on {Config.DEVICE} with K={Config.K_SHOT_COMPARISON}-Shot...")
        print("="*60)

        for name, model in models_to_test.items():
            start = time.time()

            # 1. Train AE
            trained_model = train_autoencoder(name, model, loader)

            # 2. Evaluate with MAML
            acc, f1 = evaluate_feature_extractor(name, trained_model, X_mal, X_ben)

            duration = time.time() - start
            results.append((name, acc, f1, duration))
            print(f"Done: {name} | Acc: {acc:.4f} | F1: {f1:.4f} | Time: {duration:.1f}s")

        # Final Leaderboard
        print("\n" + "="*60)
        print(f"{'Rank':<5} | {'Model':<20} | {'Accuracy':<10} | {'F1-Score':<10}")
        print("-" * 60)

        # Sort by Accuracy
        results.sort(key=lambda x: x[1], reverse=True)

        for i, (name, acc, f1, _) in enumerate(results):
            print(f"{i+1:<5} | {name:<20} | {acc:.4f}     | {f1:.4f}")
        print("="*60)

Loading dataset from /content/Dataset.csv...

Starting Comparison on cuda with K=5-Shot...
--> Training Standard AE...
Done: Standard AE | Acc: 0.7964 | F1: 0.8031 | Time: 49.4s
--> Training Denoising AE...
Done: Denoising AE | Acc: 0.8208 | F1: 0.8277 | Time: 59.4s
--> Training Sparse AE...
Done: Sparse AE | Acc: 0.8364 | F1: 0.8430 | Time: 53.6s
--> Training Variational AE...
Done: Variational AE | Acc: 0.8623 | F1: 0.8565 | Time: 62.6s

Rank  | Model                | Accuracy   | F1-Score  
------------------------------------------------------------
1     | Variational AE       | 0.8623     | 0.8565
2     | Sparse AE            | 0.8364     | 0.8430
3     | Denoising AE         | 0.8208     | 0.8277
4     | Standard AE          | 0.7964     | 0.8031


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef
import time

class Config:
    DATASET_PATH = '/content/Dataset.csv'

    INPUT_DIM = 72
    LATENT_DIM = 64
    N_WAY = 2

    # We will loop over K values, but these are defaults
    K_SHOT = 5
    Q_QUERY = 100

    AE_LR = 0.001
    AE_EPOCHS = 50

    META_LR = 0.005
    INNER_LR = 0.01
    INNER_STEPS = 5

    META_EPISODES = 500  # Reduced slightly for faster comparison loop
    TEST_EPISODES = 50

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def preprocess_dataset(filepath):
    print(f"Loading dataset from {filepath}")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print("Error: Dataset file not found.")
        return None, None

    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    # Handle Target Column strictly
    target_col = 'Class' if 'Class' in df.columns else df.columns[-1]
    X = df.drop(columns=cols_to_drop + [target_col])
    y = df[target_col]

    # Binary Encoding: Benign vs Malware
    y_enc = y.apply(lambda x: 0 if str(x).lower().strip() == 'benign' else 1).values

    X_processed = X.copy()
    numeric_cols = []
    categorical_cols = []

    for col in X_processed.columns:
        if pd.api.types.is_numeric_dtype(X_processed[col]):
            numeric_cols.append(col)
        else:
            try:
                sample = X_processed[col].dropna().iloc[0] if not X_processed[col].dropna().empty else ""
                def convert_hex(x):
                    if isinstance(x, str):
                        if x.startswith('0x') or x.startswith('0X'): return int(x, 16)
                        else: return float(x)
                    return x

                if isinstance(sample, str):
                    convert_hex(sample)
                    X_processed[col] = X_processed[col].apply(convert_hex)
                    numeric_cols.append(col)
            except (ValueError, TypeError):
                categorical_cols.append(col)

    for col in categorical_cols:
        X_processed[col] = X_processed[col].astype(str)
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col])

    scaler = MinMaxScaler()
    X_final = scaler.fit_transform(X_processed)

    # Ensure 72 features
    if X_final.shape[1] < 72:
        pad = np.zeros((X_final.shape[0], 72 - X_final.shape[1]))
        X_final = np.hstack([X_final, pad])
    elif X_final.shape[1] > 72:
        X_final = X_final[:, :72]

    return X_final, y_enc

# --- Models ---

# Standard Autoencoder
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

# Variational Autoencoder
class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super(VariationalAutoencoder, self).__init__()
        # Encoder components
        self.fc1 = nn.Linear(Config.INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, Config.LATENT_DIM)
        self.fc_logvar = nn.Linear(128, Config.LATENT_DIM)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        # Return z first to match Autoencoder signature for get_few_shot_batch
        return z, x_recon, mu, logvar

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)

# --- Logic Updates ---

def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    # Sample Support
    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    # Sample Query
    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)

    q_size = min(Config.Q_QUERY, len(remaining_mal), len(remaining_ben))

    idx_mal_q = np.random.choice(remaining_mal, q_size, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, q_size, replace=False)

    # Batches
    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]]).to(Config.DEVICE)
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long().to(Config.DEVICE)

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]]).to(Config.DEVICE)
    y_qry = torch.cat([torch.ones(q_size), torch.zeros(q_size)]).long().to(Config.DEVICE)

    # Extract Features
    with torch.no_grad():
        out_sup = ae_model(X_sup)
        out_qry = ae_model(X_qry)

        # --- CRITICAL FIX ---
        # If the model returns 4 values (z, recon, mu, logvar), it is a VAE.
        # We must use 'mu' (index 2) for classification, not 'z' (index 0).
        if isinstance(out_sup, tuple) and len(out_sup) == 4:
            z_sup = out_sup[2] # Use Mean (Stable)
            z_qry = out_qry[2] # Use Mean (Stable)
        else:
            z_sup = out_sup[0] # Standard/Denoising/Sparse (Use Z)
            z_qry = out_qry[0]

    return z_sup, y_sup, z_qry, y_qry

def manual_maml_step(model, z_sup, y_sup, z_qry, y_qry, train=True):
    fast_weights = dict(model.named_parameters())

    # Inner Loop
    for _ in range(Config.INNER_STEPS):
        out = torch.func.functional_call(model, fast_weights, z_sup)
        loss = nn.CrossEntropyLoss()(out, y_sup)
        grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=train)
        fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

    # Outer Loop
    out_q = torch.func.functional_call(model, fast_weights, z_qry)

    if train:
        meta_loss = nn.CrossEntropyLoss()(out_q, y_qry)
        return meta_loss
    else:
        preds = torch.argmax(out_q, dim=1)
        return preds, y_qry

def calculate_metrics(y_true, y_pred):
    y_true = y_true.cpu().numpy()
    y_pred = y_pred.cpu().numpy()
    return (accuracy_score(y_true, y_pred),
            f1_score(y_true, y_pred, average='binary'),
            precision_score(y_true, y_pred, average='binary'),
            recall_score(y_true, y_pred, average='binary'),
            matthews_corrcoef(y_true, y_pred))

# --- Main Execution Loop ---

if __name__ == "__main__":
    X_np, y_np = preprocess_dataset(Config.DATASET_PATH)

    if X_np is None:
        exit()

    X_tensor = torch.FloatTensor(X_np)
    y_tensor = torch.LongTensor(y_np)
    X_mal = X_tensor[y_tensor == 1]
    X_ben = X_tensor[y_tensor == 0]

    # Define the 4 models to compare
    models_config = [
        ("Standard Autoencoder", Autoencoder),
        ("Denoising Autoencoder", Autoencoder),  # Same class, different training logic
        ("Sparse Autoencoder", Autoencoder),     # Same class, different loss logic
        ("Variational Autoencoder", VariationalAutoencoder)
    ]

    final_results = []

    print(f"\n{'='*20} STARTING COMPARISON {'='*20}")

    for model_name, ModelClass in models_config:
        print(f"\n>> Evaluating: {model_name}")

        # 1. Initialize Model
        ae = ModelClass().to(Config.DEVICE)
        opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

        # 2. Train AE (with specific logic per type)
        ae.train()
        for epoch in range(Config.AE_EPOCHS):
            for (batch_x,) in loader:
                batch_x = batch_x.to(Config.DEVICE)
                opt_ae.zero_grad()

                if model_name == "Denoising Autoencoder":
                    # Add noise to input
                    noise = torch.randn_like(batch_x) * 0.1
                    noisy_x = batch_x + noise
                    z, recon = ae(noisy_x)
                    loss = nn.MSELoss()(recon, batch_x) # Reconstruct original

                elif model_name == "Sparse Autoencoder":
                    # Add L1 penalty on Z
                    z, recon = ae(batch_x)
                    mse = nn.MSELoss()(recon, batch_x)
                    l1_loss = torch.mean(torch.abs(z))
                    loss = mse + (0.0001 * l1_loss)

                elif model_name == "Variational Autoencoder":
                    # MSE + KL Divergence
                    z, recon, mu, logvar = ae(batch_x)
                    mse = nn.MSELoss(reduction='sum')(recon, batch_x)
                    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = (mse + kld * 0.001) / batch_x.size(0)

                else: # Standard
                    z, recon = ae(batch_x)
                    loss = nn.MSELoss()(recon, batch_x)

                loss.backward()
                opt_ae.step()

        ae.eval()
        print(f"   {model_name} Training Complete.")

        # 3. Evaluate using MAML (Fixed K=5 for comparison fairness)
        eval_k = 5
        Config.K_SHOT = eval_k

        maml = MAMLClassifier().to(Config.DEVICE)
        meta_optimizer = optim.Adam(maml.parameters(), lr=Config.META_LR)

        # Meta-Training
        maml.train()
        for episode in range(Config.META_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae, eval_k)
            meta_optimizer.zero_grad()
            loss = manual_maml_step(maml, z_s, y_s, z_q, y_q, train=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(maml.parameters(), 1.0)
            meta_optimizer.step()

        # Meta-Testing
        metrics_accum = {"Acc": [], "F1": []}
        for _ in range(Config.TEST_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae, eval_k)
            preds, targets = manual_maml_step(maml, z_s, y_s, z_q, y_q, train=False)
            acc, f1, _, _, _ = calculate_metrics(targets, preds)
            metrics_accum["Acc"].append(acc)
            metrics_accum["F1"].append(f1)

        avg_acc = np.mean(metrics_accum["Acc"])
        avg_f1 = np.mean(metrics_accum["F1"])

        final_results.append({
            "Model": model_name,
            "Accuracy": avg_acc,
            "F1-Score": avg_f1
        })
        print(f"   Result: Acc={avg_acc:.4f}, F1={avg_f1:.4f}")

    # --- Final Leaderboard ---
    print(f"\n{'='*20} FINAL LEADERBOARD (K=5) {'='*20}")
    print(f"{'Model':<25} | {'Accuracy':<10} | {'F1-Score':<10}")
    print("-" * 50)

    # Sort results by Accuracy
    final_results.sort(key=lambda x: x['Accuracy'], reverse=True)

    for res in final_results:
        print(f"{res['Model']:<25} | {res['Accuracy']:.4f}     | {res['F1-Score']:.4f}")

Loading dataset from /content/Dataset.csv

==================== STARTING COMPARISON ====================

>> Evaluating: Standard Autoencoder
   Standard Autoencoder Training Complete.
   Result: Acc=0.8294, F1=0.8293

>> Evaluating: Denoising Autoencoder
   Denoising Autoencoder Training Complete.
   Result: Acc=0.8211, F1=0.8230

>> Evaluating: Sparse Autoencoder
   Sparse Autoencoder Training Complete.
   Result: Acc=0.8419, F1=0.8481

>> Evaluating: Variational Autoencoder
   Variational Autoencoder Training Complete.
   Result: Acc=0.8672, F1=0.8620

==================== FINAL LEADERBOARD (K=5) ====================
Model                     | Accuracy   | F1-Score  
--------------------------------------------------
Variational Autoencoder   | 0.8672     | 0.8620
Sparse Autoencoder        | 0.8419     | 0.8481
Standard Autoencoder      | 0.8294     | 0.8293
Denoising Autoencoder     | 0.8211     | 0.8230


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
import time

# --- Configuration (UPDATED) ---
class Config:
    DATASET_PATH = '/content/Dataset.csv' # Ensure correct path

    INPUT_DIM = 72
    LATENT_DIM = 64
    N_WAY = 2
    K_SHOT = 5
    Q_QUERY = 100

    AE_LR = 0.001
    AE_EPOCHS = 50

    META_LR = 0.005
    INNER_LR = 0.01
    INNER_STEPS = 5

    # --- ADDED THESE NEW PARAMETERS FOR RELATION NET ---
    RELATION_LR = 0.001
    RELATION_EPISODES = 500

    META_EPISODES = 500
    TEST_EPISODES = 100

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Preprocessing ---
def preprocess_dataset(filepath):
    print(f"Loading dataset from {filepath}...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print("Error: Dataset file not found.")
        return None, None

    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    target_col = 'Class' if 'Class' in df.columns else df.columns[-1]
    X = df.drop(columns=cols_to_drop + [target_col])
    y = df[target_col]

    y_enc = y.apply(lambda x: 0 if str(x).lower().strip() == 'benign' else 1).values

    X_processed = X.copy()
    numeric_cols = []
    categorical_cols = []

    for col in X_processed.columns:
        if pd.api.types.is_numeric_dtype(X_processed[col]):
            numeric_cols.append(col)
        else:
            try:
                sample = X_processed[col].dropna().iloc[0] if not X_processed[col].dropna().empty else ""
                def convert_hex(x):
                    if isinstance(x, str):
                        if x.startswith('0x') or x.startswith('0X'): return int(x, 16)
                        else: return float(x)
                    return x
                if isinstance(sample, str):
                    convert_hex(sample)
                    X_processed[col] = X_processed[col].apply(convert_hex)
                    numeric_cols.append(col)
            except:
                categorical_cols.append(col)

    for col in categorical_cols:
        X_processed[col] = X_processed[col].astype(str)
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col])

    scaler = MinMaxScaler()
    X_final = scaler.fit_transform(X_processed)

    if X_final.shape[1] < 72:
        pad = np.zeros((X_final.shape[0], 72 - X_final.shape[1]))
        X_final = np.hstack([X_final, pad])
    elif X_final.shape[1] > 72:
        X_final = X_final[:, :72]

    return X_final, y_enc

# --- Encoders ---

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super(VariationalAutoencoder, self).__init__()
        self.fc1 = nn.Linear(Config.INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, Config.LATENT_DIM)
        self.fc_logvar = nn.Linear(128, Config.LATENT_DIM)
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return z, x_recon, mu, logvar

# --- Classifiers ---

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)

class RelationModule(nn.Module):
    def __init__(self, input_dim):
        super(RelationModule, self).__init__()
        # Input dim is 2 * latent_dim (concatenation)
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        self.layer2 = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

# --- Data Sampler ---

def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)
    q_size = min(Config.Q_QUERY, len(remaining_mal), len(remaining_ben))

    idx_mal_q = np.random.choice(remaining_mal, q_size, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, q_size, replace=False)

    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]]).to(Config.DEVICE)
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long().to(Config.DEVICE)

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]]).to(Config.DEVICE)
    y_qry = torch.cat([torch.ones(q_size), torch.zeros(q_size)]).long().to(Config.DEVICE)

    with torch.no_grad():
        out_sup = ae_model(X_sup)
        out_qry = ae_model(X_qry)

        # VAE Check (Tuple len 4) vs AE Check (Tuple len 2)
        if isinstance(out_sup, tuple) and len(out_sup) == 4:
            z_sup = out_sup[2] # Use MU
            z_qry = out_qry[2] # Use MU
        else:
            z_sup = out_sup[0]
            z_qry = out_qry[0]

    return z_sup, y_sup, z_qry, y_qry

# --- Evaluation Routines ---

def run_maml(ae_model, X_mal, X_ben):
    maml = MAMLClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(maml.parameters(), lr=Config.META_LR)

    maml.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        meta_loss = nn.CrossEntropyLoss()(out_q, y_q)
        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values())
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_protonet(ae_model, X_mal, X_ben):
    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        prototypes = []
        for class_idx in range(Config.N_WAY):
            class_mask = (y_s == class_idx)
            class_samples = z_s[class_mask]
            if len(class_samples) > 0: prototypes.append(class_samples.mean(dim=0))
            else: prototypes.append(torch.zeros_like(z_s[0]))
        prototypes = torch.stack(prototypes)

        dists = torch.cdist(z_q, prototypes)
        preds = torch.argmin(dists, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_relation_network(ae_model, X_mal, X_ben):
    relation_net = RelationModule(input_dim=Config.LATENT_DIM).to(Config.DEVICE)
    optimizer = optim.Adam(relation_net.parameters(), lr=Config.RELATION_LR)
    loss_fn = nn.MSELoss()

    relation_net.train()
    for episode in range(Config.RELATION_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)
        optimizer.zero_grad()

        prototypes = []
        for class_idx in range(Config.N_WAY):
            mask = (y_s == class_idx)
            class_samples = z_s[mask]
            if len(class_samples) > 0: prototypes.append(class_samples.mean(dim=0))
            else: prototypes.append(torch.zeros_like(z_s[0]))
        prototypes = torch.stack(prototypes)

        n_queries = z_q.size(0)
        n_classes = Config.N_WAY

        proto_ext = prototypes.unsqueeze(0).repeat(n_queries, 1, 1).view(-1, Config.LATENT_DIM)
        query_ext = z_q.unsqueeze(1).repeat(1, n_classes, 1).view(-1, Config.LATENT_DIM)
        relation_pairs = torch.cat([proto_ext, query_ext], dim=1)

        scores = relation_net(relation_pairs).view(n_queries, n_classes)
        target_scores = torch.zeros(n_queries, n_classes).to(Config.DEVICE)
        target_scores.scatter_(1, y_q.unsqueeze(1), 1)

        loss = loss_fn(scores, target_scores)
        loss.backward()
        optimizer.step()

    relation_net.eval()
    accs, f1s = [], []
    with torch.no_grad():
        for _ in range(Config.TEST_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

            prototypes = []
            for class_idx in range(Config.N_WAY):
                mask = (y_s == class_idx)
                class_samples = z_s[mask]
                if len(class_samples) > 0: prototypes.append(class_samples.mean(dim=0))
                else: prototypes.append(torch.zeros_like(z_s[0]))
            prototypes = torch.stack(prototypes)

            n_queries = z_q.size(0)
            n_classes = Config.N_WAY

            proto_ext = prototypes.unsqueeze(0).repeat(n_queries, 1, 1).view(-1, Config.LATENT_DIM)
            query_ext = z_q.unsqueeze(1).repeat(1, n_classes, 1).view(-1, Config.LATENT_DIM)
            relation_pairs = torch.cat([proto_ext, query_ext], dim=1)

            scores = relation_net(relation_pairs).view(n_queries, n_classes)
            preds = torch.argmax(scores, dim=1).cpu().numpy()
            targets = y_q.cpu().numpy()

            accs.append(accuracy_score(targets, preds))
            f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

# --- Main Execution ---

if __name__ == "__main__":
    X_np, y_np = preprocess_dataset(Config.DATASET_PATH)
    if X_np is None: exit()

    X_tensor = torch.FloatTensor(X_np)
    y_tensor = torch.LongTensor(y_np)
    X_mal = X_tensor[y_tensor == 1]
    X_ben = X_tensor[y_tensor == 0]

    encoders = [
        ("Sparse AE", Autoencoder, "sparse"),
        ("Variational AE", VariationalAutoencoder, "vae")
    ]

    classifiers = ["MAML", "ProtoNet", "RelationNet"]

    results = []

    print(f"\n{'='*25} EXTENDED COMPARISON {'='*25}")

    for enc_name, EncClass, logic in encoders:
        print(f"\n>>> Training Encoder: {enc_name}...")
        ae = EncClass().to(Config.DEVICE)
        opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

        ae.train()
        for epoch in range(Config.AE_EPOCHS):
            for (batch_x,) in loader:
                batch_x = batch_x.to(Config.DEVICE)
                opt_ae.zero_grad()

                if logic == "sparse":
                    z, recon = ae(batch_x)
                    mse = nn.MSELoss()(recon, batch_x)
                    l1 = torch.mean(torch.abs(z))
                    loss = mse + (0.0001 * l1)
                elif logic == "vae":
                    z, recon, mu, logvar = ae(batch_x)
                    mse = nn.MSELoss(reduction='sum')(recon, batch_x)
                    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = (mse + kld * 0.001) / batch_x.size(0)

                loss.backward()
                opt_ae.step()

        ae.eval()
        print(f"    Encoder Trained.")

        for clf_name in classifiers:
            print(f"    -> Running {clf_name}...")
            start = time.time()
            if clf_name == "MAML":
                acc, f1 = run_maml(ae, X_mal, X_ben)
            elif clf_name == "ProtoNet":
                acc, f1 = run_protonet(ae, X_mal, X_ben)
            elif clf_name == "RelationNet":
                acc, f1 = run_relation_network(ae, X_mal, X_ben)

            duration = time.time() - start
            results.append({
                "Combination": f"{enc_name} + {clf_name}",
                "Accuracy": acc,
                "F1-Score": f1,
                "Time": duration
            })

    print(f"\n{'='*20} FINAL LEADERBOARD {'='*20}")
    print(f"{'Combination':<30} | {'Accuracy':<10} | {'F1-Score':<10} | {'Time (s)':<10}")
    print("-" * 65)

    results.sort(key=lambda x: x['Accuracy'], reverse=True)
    for res in results:
        print(f"{res['Combination']:<30} | {res['Accuracy']:.4f}     | {res['F1-Score']:.4f}     | {res['Time']:.1f}")

Loading dataset from /content/Dataset.csv...

========================= EXTENDED COMPARISON =========================

>>> Training Encoder: Sparse AE...
    Encoder Trained.
    -> Running MAML...
    -> Running ProtoNet...
    -> Running RelationNet...

>>> Training Encoder: Variational AE...
    Encoder Trained.
    -> Running MAML...
    -> Running ProtoNet...
    -> Running RelationNet...

==================== FINAL LEADERBOARD ====================
Combination                    | Accuracy   | F1-Score   | Time (s)  
-----------------------------------------------------------------
Variational AE + MAML          | 0.8774     | 0.8715     | 9.3
Sparse AE + MAML               | 0.8420     | 0.8403     | 9.8
Variational AE + RelationNet   | 0.8311     | 0.8319     | 3.3
Sparse AE + RelationNet        | 0.8004     | 0.8110     | 2.8
Sparse AE + ProtoNet           | 0.6212     | 0.6300     | 0.6
Variational AE + ProtoNet      | 0.6074     | 0.6287     | 0.6


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
import time

# --- Configuration ---
class Config:
    DATASET_PATH = '/content/Dataset.csv' # Ensure correct path

    INPUT_DIM = 72
    LATENT_DIM = 64
    N_WAY = 2
    K_SHOT = 5
    Q_QUERY = 100

    AE_LR = 0.001
    AE_EPOCHS = 50

    META_LR = 0.005 # Beta

    # MAML / MAML++ Params
    INNER_LR = 0.01 # Initial Alpha
    INNER_STEPS = 5

    # CAVIA Params
    CAVIA_CONTEXT_DIM = 50 # Size of the context vector
    CAVIA_INNER_LR = 0.1   # Learning rate for context vector (usually higher)

    META_EPISODES = 500
    TEST_EPISODES = 100

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Preprocessing (Unchanged) ---
def preprocess_dataset(filepath):
    print(f"Loading dataset from {filepath}...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print("Error: Dataset file not found.")
        return None, None

    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    target_col = 'Class' if 'Class' in df.columns else df.columns[-1]
    X = df.drop(columns=cols_to_drop + [target_col])
    y = df[target_col]

    y_enc = y.apply(lambda x: 0 if str(x).lower().strip() == 'benign' else 1).values

    X_processed = X.copy()
    numeric_cols = []
    categorical_cols = []

    for col in X_processed.columns:
        if pd.api.types.is_numeric_dtype(X_processed[col]):
            numeric_cols.append(col)
        else:
            try:
                sample = X_processed[col].dropna().iloc[0] if not X_processed[col].dropna().empty else ""
                def convert_hex(x):
                    if isinstance(x, str):
                        if x.startswith('0x') or x.startswith('0X'): return int(x, 16)
                        else: return float(x)
                    return x
                if isinstance(sample, str):
                    convert_hex(sample)
                    X_processed[col] = X_processed[col].apply(convert_hex)
                    numeric_cols.append(col)
            except:
                categorical_cols.append(col)

    for col in categorical_cols:
        X_processed[col] = X_processed[col].astype(str)
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col])

    scaler = MinMaxScaler()
    X_final = scaler.fit_transform(X_processed)

    if X_final.shape[1] < 72:
        pad = np.zeros((X_final.shape[0], 72 - X_final.shape[1]))
        X_final = np.hstack([X_final, pad])
    elif X_final.shape[1] > 72:
        X_final = X_final[:, :72]

    return X_final, y_enc

# --- Encoders (Unchanged) ---

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super(VariationalAutoencoder, self).__init__()
        self.fc1 = nn.Linear(Config.INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, Config.LATENT_DIM)
        self.fc_logvar = nn.Linear(128, Config.LATENT_DIM)
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return z, x_recon, mu, logvar

# --- Classifiers (MAML & CAVIA) ---

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)

class CAVIAClassifier(nn.Module):
    def __init__(self):
        super(CAVIAClassifier, self).__init__()
        # Input is Latent Features + Context Vector
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM + Config.CAVIA_CONTEXT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x, context):
        # Expand context to match batch size: (1, Context_Dim) -> (Batch, Context_Dim)
        batch_size = x.size(0)
        context_expanded = context.expand(batch_size, -1)

        # Concatenate: [Features, Context]
        combined = torch.cat([x, context_expanded], dim=1)
        return self.net(combined)

# --- Data Sampler (Unchanged) ---

def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)
    q_size = min(Config.Q_QUERY, len(remaining_mal), len(remaining_ben))

    idx_mal_q = np.random.choice(remaining_mal, q_size, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, q_size, replace=False)

    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]]).to(Config.DEVICE)
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long().to(Config.DEVICE)

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]]).to(Config.DEVICE)
    y_qry = torch.cat([torch.ones(q_size), torch.zeros(q_size)]).long().to(Config.DEVICE)

    with torch.no_grad():
        out_sup = ae_model(X_sup)
        out_qry = ae_model(X_qry)

        if isinstance(out_sup, tuple) and len(out_sup) == 4:
            z_sup = out_sup[2] # Use MU for VAE
            z_qry = out_qry[2]
        else:
            z_sup = out_sup[0] # Use Z for AE
            z_qry = out_qry[0]

    return z_sup, y_sup, z_qry, y_qry

# --- Evaluation Routines ---

def run_maml(ae_model, X_mal, X_ben):
    maml = MAMLClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(maml.parameters(), lr=Config.META_LR)

    maml.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        meta_loss = nn.CrossEntropyLoss()(out_q, y_q)
        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values())
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_maml_plus_plus(ae_model, X_mal, X_ben):
    """
    MAML++ Simplified Implementation:
    1. Learnable Per-Layer Learning Rates (LSLR)
    2. Multi-Step Loss Optimization (MSL) - Optimizes weighted loss over all steps
    """
    maml = MAMLClassifier().to(Config.DEVICE)

    # Initialize Learnable Learning Rates (same shape as params)
    # Start small (e.g., Config.INNER_LR)
    learning_rates = {}
    for name, param in maml.named_parameters():
        lr_tensor = torch.tensor(Config.INNER_LR).to(Config.DEVICE)
        lr_tensor.requires_grad = True
        learning_rates[name] = lr_tensor

    # Optimize both model weights and learning rates
    meta_params = list(maml.parameters()) + list(learning_rates.values())
    meta_opt = optim.Adam(meta_params, lr=Config.META_LR)

    maml.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        query_losses = [] # For Multi-Step Loss

        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)

            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)

            # Update using Learnable Learning Rates
            fast_weights = {
                n: p - learning_rates[n] * g
                for (n, p), g in zip(fast_weights.items(), grads)
            }

            # Compute Query Loss at this step (for MSL)
            out_q_step = torch.func.functional_call(maml, fast_weights, z_q)
            loss_q_step = nn.CrossEntropyLoss()(out_q_step, y_q)
            query_losses.append(loss_q_step)

        # MAML++ Multi-Step Loss: Weighted sum of query losses
        # Assign increasing weights to later steps
        weights = [0.1 * (i+1) for i in range(Config.INNER_STEPS)]
        total_weights = sum(weights)
        weighted_loss = sum(w * l for w, l in zip(weights, query_losses)) / total_weights

        meta_opt.zero_grad()
        weighted_loss.backward()
        meta_opt.step()

    # Evaluation
    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values())
            # Use learned LRs (detached)
            fast_weights = {
                n: p - learning_rates[n].detach() * g
                for (n, p), g in zip(fast_weights.items(), grads)
            }

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_cavia(ae_model, X_mal, X_ben):
    """
    CAVIA: Fast Context Adaptation.
    Inner loop optimizes a 'Context Vector' instead of model weights.
    """
    cavia = CAVIAClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(cavia.parameters(), lr=Config.META_LR)

    cavia.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        # Initialize Context Parameters (Learned per task, starting at 0)
        # Shape: (1, Context_Dim) - Broadcasted later
        context_params = torch.zeros(1, Config.CAVIA_CONTEXT_DIM, device=Config.DEVICE, requires_grad=True)

        # Inner Loop: Optimize Context Only
        # Using SGD for inner loop on context
        for _ in range(Config.INNER_STEPS):
            # Forward pass using context
            out = cavia(z_s, context_params)
            loss = nn.CrossEntropyLoss()(out, y_s)

            # Gradients for Context
            grad = torch.autograd.grad(loss, context_params, create_graph=True)[0]
            context_params = context_params - Config.CAVIA_INNER_LR * grad

        # Outer Loop: Optimize Network Weights
        # Forward pass using ADAPTED context
        out_q = cavia(z_q, context_params)
        meta_loss = nn.CrossEntropyLoss()(out_q, y_q)

        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    # Evaluation
    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, Config.K_SHOT)

        context_params = torch.zeros(1, Config.CAVIA_CONTEXT_DIM, device=Config.DEVICE, requires_grad=True)

        for _ in range(Config.INNER_STEPS):
            out = cavia(z_s, context_params)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grad = torch.autograd.grad(loss, context_params)[0]
            context_params = context_params - Config.CAVIA_INNER_LR * grad

        out_q = cavia(z_q, context_params)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

# --- Main Execution ---

if __name__ == "__main__":
    X_np, y_np = preprocess_dataset(Config.DATASET_PATH)
    if X_np is None: exit()

    X_tensor = torch.FloatTensor(X_np)
    y_tensor = torch.LongTensor(y_np)
    X_mal = X_tensor[y_tensor == 1]
    X_ben = X_tensor[y_tensor == 0]

    encoders = [
        ("Sparse AE", Autoencoder, "sparse"),
        ("Variational AE", VariationalAutoencoder, "vae")
    ]

    # Updated Classifiers List
    classifiers = ["MAML", "MAML++", "CAVIA"]

    results = []

    print(f"\n{'='*25} ADVANCED COMPARISON {'='*25}")

    for enc_name, EncClass, logic in encoders:
        print(f"\n>>> Training Encoder: {enc_name}...")
        ae = EncClass().to(Config.DEVICE)
        opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

        ae.train()
        for epoch in range(Config.AE_EPOCHS):
            for (batch_x,) in loader:
                batch_x = batch_x.to(Config.DEVICE)
                opt_ae.zero_grad()

                if logic == "sparse":
                    z, recon = ae(batch_x)
                    mse = nn.MSELoss()(recon, batch_x)
                    l1 = torch.mean(torch.abs(z))
                    loss = mse + (0.0001 * l1)
                elif logic == "vae":
                    z, recon, mu, logvar = ae(batch_x)
                    mse = nn.MSELoss(reduction='sum')(recon, batch_x)
                    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = (mse + kld * 0.001) / batch_x.size(0)

                loss.backward()
                opt_ae.step()

        ae.eval()
        print(f"    Encoder Trained.")

        for clf_name in classifiers:
            print(f"    -> Running {clf_name}...")
            start = time.time()
            if clf_name == "MAML":
                acc, f1 = run_maml(ae, X_mal, X_ben)
            elif clf_name == "MAML++":
                acc, f1 = run_maml_plus_plus(ae, X_mal, X_ben)
            elif clf_name == "CAVIA":
                acc, f1 = run_cavia(ae, X_mal, X_ben)

            duration = time.time() - start
            results.append({
                "Combination": f"{enc_name} + {clf_name}",
                "Accuracy": acc,
                "F1-Score": f1,
                "Time": duration
            })

    print(f"\n{'='*20} FINAL LEADERBOARD {'='*20}")
    print(f"{'Combination':<35} | {'Accuracy':<10} | {'F1-Score':<10} | {'Time (s)':<10}")
    print("-" * 70)

    results.sort(key=lambda x: x['Accuracy'], reverse=True)
    for res in results:
        print(f"{res['Combination']:<35} | {res['Accuracy']:.4f}     | {res['F1-Score']:.4f}     | {res['Time']:.1f}")

Loading dataset from /content/Dataset.csv...

========================= ADVANCED COMPARISON =========================

>>> Training Encoder: Sparse AE...
    Encoder Trained.
    -> Running MAML...
    -> Running MAML++...
    -> Running CAVIA...

>>> Training Encoder: Variational AE...
    Encoder Trained.
    -> Running MAML...
    -> Running MAML++...
    -> Running CAVIA...

==================== FINAL LEADERBOARD ====================
Combination                         | Accuracy   | F1-Score   | Time (s)  
----------------------------------------------------------------------
Variational AE + MAML               | 0.8856     | 0.8859     | 9.9
Variational AE + CAVIA              | 0.8768     | 0.8725     | 8.0
Variational AE + MAML++             | 0.8711     | 0.8682     | 12.0
Sparse AE + MAML                    | 0.8371     | 0.8418     | 9.7
Sparse AE + MAML++                  | 0.8312     | 0.8306     | 11.9
Sparse AE + CAVIA                   | 0.8200     | 0.8209     | 7.8


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
import time

# --- Configuration ---
class Config:
    DATASET_PATH = '/content/Dataset.csv' # Ensure correct path

    INPUT_DIM = 72
    LATENT_DIM = 64
    N_WAY = 2
    # K_SHOT will be set dynamically in the loop
    Q_QUERY = 100

    AE_LR = 0.001
    AE_EPOCHS = 50

    META_LR = 0.005 # Beta

    # MAML / MAML++ Params
    INNER_LR = 0.01 # Initial Alpha
    INNER_STEPS = 5

    # CAVIA Params
    CAVIA_CONTEXT_DIM = 50
    CAVIA_INNER_LR = 0.1

    META_EPISODES = 500
    TEST_EPISODES = 100

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Preprocessing ---
def preprocess_dataset(filepath):
    print(f"Loading dataset from {filepath}...")
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print("Error: Dataset file not found.")
        return None, None

    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    target_col = 'Class' if 'Class' in df.columns else df.columns[-1]
    X = df.drop(columns=cols_to_drop + [target_col])
    y = df[target_col]

    y_enc = y.apply(lambda x: 0 if str(x).lower().strip() == 'benign' else 1).values

    X_processed = X.copy()
    numeric_cols = []
    categorical_cols = []

    for col in X_processed.columns:
        if pd.api.types.is_numeric_dtype(X_processed[col]):
            numeric_cols.append(col)
        else:
            try:
                sample = X_processed[col].dropna().iloc[0] if not X_processed[col].dropna().empty else ""
                def convert_hex(x):
                    if isinstance(x, str):
                        if x.startswith('0x') or x.startswith('0X'): return int(x, 16)
                        else: return float(x)
                    return x
                if isinstance(sample, str):
                    convert_hex(sample)
                    X_processed[col] = X_processed[col].apply(convert_hex)
                    numeric_cols.append(col)
            except:
                categorical_cols.append(col)

    for col in categorical_cols:
        X_processed[col] = X_processed[col].astype(str)
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col])

    scaler = MinMaxScaler()
    X_final = scaler.fit_transform(X_processed)

    if X_final.shape[1] < 72:
        pad = np.zeros((X_final.shape[0], 72 - X_final.shape[1]))
        X_final = np.hstack([X_final, pad])
    elif X_final.shape[1] > 72:
        X_final = X_final[:, :72]

    return X_final, y_enc

# --- Encoders ---

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super(VariationalAutoencoder, self).__init__()
        self.fc1 = nn.Linear(Config.INPUT_DIM, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc_mu = nn.Linear(128, Config.LATENT_DIM)
        self.fc_logvar = nn.Linear(128, Config.LATENT_DIM)
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def encode(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return z, x_recon, mu, logvar

# --- Classifiers ---

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)

class CAVIAClassifier(nn.Module):
    def __init__(self):
        super(CAVIAClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM + Config.CAVIA_CONTEXT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x, context):
        batch_size = x.size(0)
        context_expanded = context.expand(batch_size, -1)
        combined = torch.cat([x, context_expanded], dim=1)
        return self.net(combined)

# --- Data Sampler ---

def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    # Ensure we have enough samples
    if len(X_mal) < k_shot + Config.Q_QUERY or len(X_ben) < k_shot + Config.Q_QUERY:
         # Fallback to smaller query size if data is scarce for large K
         current_q = min(len(X_mal) - k_shot, len(X_ben) - k_shot, Config.Q_QUERY)
         if current_q <= 0: return None, None, None, None # Should not happen with this dataset size
    else:
         current_q = Config.Q_QUERY

    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)

    idx_mal_q = np.random.choice(remaining_mal, current_q, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, current_q, replace=False)

    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]]).to(Config.DEVICE)
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long().to(Config.DEVICE)

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]]).to(Config.DEVICE)
    y_qry = torch.cat([torch.ones(current_q), torch.zeros(current_q)]).long().to(Config.DEVICE)

    with torch.no_grad():
        out_sup = ae_model(X_sup)
        out_qry = ae_model(X_qry)

        if isinstance(out_sup, tuple) and len(out_sup) == 4:
            z_sup = out_sup[2] # Use MU for VAE
            z_qry = out_qry[2]
        else:
            z_sup = out_sup[0] # Use Z for AE
            z_qry = out_qry[0]

    return z_sup, y_sup, z_qry, y_qry

# --- Evaluation Routines ---

def run_maml(ae_model, X_mal, X_ben, k_shot):
    maml = MAMLClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(maml.parameters(), lr=Config.META_LR)

    maml.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        meta_loss = nn.CrossEntropyLoss()(out_q, y_q)
        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values())
            fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_maml_plus_plus(ae_model, X_mal, X_ben, k_shot):
    maml = MAMLClassifier().to(Config.DEVICE)

    learning_rates = {}
    for name, param in maml.named_parameters():
        lr_tensor = torch.tensor(Config.INNER_LR).to(Config.DEVICE)
        lr_tensor.requires_grad = True
        learning_rates[name] = lr_tensor

    meta_params = list(maml.parameters()) + list(learning_rates.values())
    meta_opt = optim.Adam(meta_params, lr=Config.META_LR)

    maml.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        fast_weights = dict(maml.named_parameters())
        query_losses = []

        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = {n: p - learning_rates[n] * g for (n, p), g in zip(fast_weights.items(), grads)}

            out_q_step = torch.func.functional_call(maml, fast_weights, z_q)
            loss_q_step = nn.CrossEntropyLoss()(out_q_step, y_q)
            query_losses.append(loss_q_step)

        weights = [0.1 * (i+1) for i in range(Config.INNER_STEPS)]
        total_weights = sum(weights)
        weighted_loss = sum(w * l for w, l in zip(weights, query_losses)) / total_weights

        meta_opt.zero_grad()
        weighted_loss.backward()
        meta_opt.step()

    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        fast_weights = dict(maml.named_parameters())
        for _ in range(Config.INNER_STEPS):
            out = torch.func.functional_call(maml, fast_weights, z_s)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grads = torch.autograd.grad(loss, fast_weights.values())
            fast_weights = {n: p - learning_rates[n].detach() * g for (n, p), g in zip(fast_weights.items(), grads)}

        out_q = torch.func.functional_call(maml, fast_weights, z_q)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

def run_cavia(ae_model, X_mal, X_ben, k_shot):
    cavia = CAVIAClassifier().to(Config.DEVICE)
    meta_opt = optim.Adam(cavia.parameters(), lr=Config.META_LR)

    cavia.train()
    for episode in range(Config.META_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        context_params = torch.zeros(1, Config.CAVIA_CONTEXT_DIM, device=Config.DEVICE, requires_grad=True)

        for _ in range(Config.INNER_STEPS):
            out = cavia(z_s, context_params)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grad = torch.autograd.grad(loss, context_params, create_graph=True)[0]
            context_params = context_params - Config.CAVIA_INNER_LR * grad

        out_q = cavia(z_q, context_params)
        meta_loss = nn.CrossEntropyLoss()(out_q, y_q)

        meta_opt.zero_grad()
        meta_loss.backward()
        meta_opt.step()

    accs, f1s = [], []
    for _ in range(Config.TEST_EPISODES):
        z_s, y_s, z_q, y_q = get_few_shot_batch(X_mal, X_ben, ae_model, k_shot)
        if z_s is None: continue

        context_params = torch.zeros(1, Config.CAVIA_CONTEXT_DIM, device=Config.DEVICE, requires_grad=True)

        for _ in range(Config.INNER_STEPS):
            out = cavia(z_s, context_params)
            loss = nn.CrossEntropyLoss()(out, y_s)
            grad = torch.autograd.grad(loss, context_params)[0]
            context_params = context_params - Config.CAVIA_INNER_LR * grad

        out_q = cavia(z_q, context_params)
        preds = torch.argmax(out_q, dim=1).cpu().numpy()
        targets = y_q.cpu().numpy()
        accs.append(accuracy_score(targets, preds))
        f1s.append(f1_score(targets, preds))

    return np.mean(accs), np.mean(f1s)

# --- Main Execution ---

if __name__ == "__main__":
    X_np, y_np = preprocess_dataset(Config.DATASET_PATH)
    if X_np is None: exit()

    X_tensor = torch.FloatTensor(X_np)
    y_tensor = torch.LongTensor(y_np)
    X_mal = X_tensor[y_tensor == 1]
    X_ben = X_tensor[y_tensor == 0]

    encoders = [
        ("Sparse AE", Autoencoder, "sparse"),
        ("Variational AE", VariationalAutoencoder, "vae")
    ]

    classifiers = ["MAML", "MAML++", "CAVIA"]

    k_shot_values = [1, 2, 5, 10, 20, 50]

    print(f"\n{'='*30} FULL K-SHOT ANALYSIS {'='*30}")

    # Store results for final table
    final_results = []

    for enc_name, EncClass, logic in encoders:
        print(f"\n>>> Training Base Encoder: {enc_name}...")
        ae = EncClass().to(Config.DEVICE)
        opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)

        ae.train()
        for epoch in range(Config.AE_EPOCHS):
            for (batch_x,) in loader:
                batch_x = batch_x.to(Config.DEVICE)
                opt_ae.zero_grad()
                if logic == "sparse":
                    z, recon = ae(batch_x)
                    mse = nn.MSELoss()(recon, batch_x)
                    l1 = torch.mean(torch.abs(z))
                    loss = mse + (0.0001 * l1)
                elif logic == "vae":
                    z, recon, mu, logvar = ae(batch_x)
                    mse = nn.MSELoss(reduction='sum')(recon, batch_x)
                    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = (mse + kld * 0.001) / batch_x.size(0)
                loss.backward()
                opt_ae.step()
        ae.eval()

        for clf_name in classifiers:
            print(f"    -> Evaluating {clf_name} across K-Shots...")
            for k in k_shot_values:
                start = time.time()
                if clf_name == "MAML":
                    acc, f1 = run_maml(ae, X_mal, X_ben, k)
                elif clf_name == "MAML++":
                    acc, f1 = run_maml_plus_plus(ae, X_mal, X_ben, k)
                elif clf_name == "CAVIA":
                    acc, f1 = run_cavia(ae, X_mal, X_ben, k)

                duration = time.time() - start

                result_entry = {
                    "Encoder": enc_name,
                    "Classifier": clf_name,
                    "K-Shot": k,
                    "Accuracy": acc,
                    "F1-Score": f1,
                    "Time": duration
                }
                final_results.append(result_entry)
                print(f"       [K={k}] Acc: {acc:.4f} | F1: {f1:.4f} | {duration:.1f}s")

    # --- Print Final Summary Table ---
    print(f"\n{'='*40} FINAL SUMMARY TABLE {'='*40}")
    print(f"{'Encoder':<15} | {'Classifier':<10} | {'K':<3} | {'Acc':<8} | {'F1':<8} | {'Time'}")
    print("-" * 65)

    # Sort for easy reading
    final_results.sort(key=lambda x: (x['Encoder'], x['Classifier'], x['K-Shot']))

    for res in final_results:
        print(f"{res['Encoder']:<15} | {res['Classifier']:<10} | {res['K-Shot']:<3} | {res['Accuracy']:.4f}   | {res['F1-Score']:.4f}   | {res['Time']:.1f}s")

Loading dataset from /content/Dataset.csv...

============================== FULL K-SHOT ANALYSIS ==============================

>>> Training Base Encoder: Sparse AE...
    -> Evaluating MAML across K-Shots...
       [K=1] Acc: 0.8390 | F1: 0.8374 | 9.7s
       [K=2] Acc: 0.8372 | F1: 0.8424 | 10.0s
       [K=5] Acc: 0.8463 | F1: 0.8474 | 9.9s
       [K=10] Acc: 0.8617 | F1: 0.8601 | 10.1s
       [K=20] Acc: 0.8671 | F1: 0.8635 | 10.1s
       [K=50] Acc: 0.8812 | F1: 0.8798 | 9.6s
    -> Evaluating MAML++ across K-Shots...
       [K=1] Acc: 0.8499 | F1: 0.8427 | 11.8s
       [K=2] Acc: 0.8564 | F1: 0.8578 | 11.9s
       [K=5] Acc: 0.8461 | F1: 0.8467 | 11.9s
       [K=10] Acc: 0.8616 | F1: 0.8624 | 12.1s
       [K=20] Acc: 0.8696 | F1: 0.8703 | 12.0s
       [K=50] Acc: 0.8235 | F1: 0.8321 | 12.0s
    -> Evaluating CAVIA across K-Shots...
       [K=1] Acc: 0.8399 | F1: 0.8478 | 7.9s
       [K=2] Acc: 0.8068 | F1: 0.8120 | 7.9s
       [K=5] Acc: 0.8476 | F1: 0.8399 | 7.3s
       [K=10] 

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, matthews_corrcoef, confusion_matrix)
import time

class Config:
    # Dataset path relative to this script's directory
    DATASET_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'Dataset.csv')

    INPUT_DIM = 72       # 72 PE/behavioral features after dropping identifiers
    LATENT_DIM = 64      # Autoencoder bottleneck dimension (Table II)
    N_WAY = 2            # Binary classification: malware vs benign

    K_SHOT = 1           # Starting K-shot value (varied: 1,2,5,10,20,50)
    Q_QUERY = 1000       # Query samples per class (Table IV: nquery=2000 total)

    AE_LR = 0.001        # Autoencoder learning rate
    AE_EPOCHS = 100      # Autoencoder training epochs
    AE_BATCH_SIZE = 64   # Autoencoder training batch size

    META_LR = 0.001      # Outer-loop meta-learning rate beta (Section III)
    INNER_LR = 0.01      # Inner-loop learning rate alpha (Section III)
    INNER_STEPS = 5      # Inner-loop gradient steps (Section III)

    META_EPISODES = 1000 # Number of meta-training episodes per K
    TEST_EPISODES = 50   # Number of meta-testing episodes per K

    SEED = 42            # Random seed for reproducibility
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    """Set random seeds for reproducibility across all libraries."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def convert_hex(x):
    """Convert hex strings (e.g., '0x108ec') to integers, numeric strings to
    floats, and pass through numeric values unchanged."""
    if isinstance(x, str):
        x_stripped = x.strip()
        if x_stripped.startswith('0x') or x_stripped.startswith('0X'):
            return int(x_stripped, 16)
        else:
            return float(x_stripped)
    return x


def preprocess_dataset(filepath):
    """Load and preprocess the Ransomware Dataset 2024.

    Preprocessing steps (Section II-A):
      1. Drop identifier columns (md5, sha1, Category, Family).
      2. Convert hex-encoded numeric strings to integers.
      3. Label-encode remaining categorical features.
      4. Apply Min-Max normalization to [0, 1] (Eq. 1).
    """
    print(f"Loading dataset from {filepath}")
    df = pd.read_csv(filepath)
    print(f"  Dataset shape: {df.shape}")
    print(f"  Class distribution:\n{df['Class'].value_counts().to_string()}")

    # Drop identifier / metadata columns that are not part of the 72 features
    cols_to_drop = ['md5', 'sha1', 'Category', 'Family']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]

    X = df.drop(columns=cols_to_drop + ['Class'])
    y = df['Class']

    # Encode target: Benign=0, Malware=1
    le_y = LabelEncoder()
    y_enc = le_y.fit_transform(y)
    print(f"  Label mapping: {dict(zip(le_y.classes_, le_y.transform(le_y.classes_)))}")

    X_processed = X.copy()
    numeric_cols = []
    categorical_cols = []

    for col in X_processed.columns:
        if pd.api.types.is_numeric_dtype(X_processed[col]):
            numeric_cols.append(col)
        else:
            # Attempt hex / numeric conversion
            try:
                X_processed[col] = X_processed[col].apply(convert_hex)
                numeric_cols.append(col)
            except (ValueError, TypeError):
                categorical_cols.append(col)

    # Label-encode any remaining categorical features (Section II-A)
    for col in categorical_cols:
        X_processed[col] = X_processed[col].astype(str)
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_processed[col])

    print(f"  Numeric features: {len(numeric_cols)}, Categorical (label-encoded): {len(categorical_cols)}")
    print(f"  Final feature count: {X_processed.shape[1]}")

    # NOTE: Min-Max normalization is deferred to after the train/test split
    # to prevent data leakage (scaler must be fit on training data only).
    return X_processed.values.astype(np.float32), y_enc, le_y


class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(Config.INPUT_DIM, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, Config.LATENT_DIM), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, Config.INPUT_DIM)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return z, x_recon

class MAMLClassifier(nn.Module):
    def __init__(self):
        super(MAMLClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(Config.LATENT_DIM, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, Config.N_WAY)
        )

    def forward(self, x):
        return self.net(x)


def get_few_shot_batch(X_mal, X_ben, ae_model, k_shot):
    # Sample Support
    idx_mal_s = np.random.choice(len(X_mal), k_shot, replace=False)
    idx_ben_s = np.random.choice(len(X_ben), k_shot, replace=False)

    # Sample Query
    remaining_mal = np.setdiff1d(np.arange(len(X_mal)), idx_mal_s)
    remaining_ben = np.setdiff1d(np.arange(len(X_ben)), idx_ben_s)

    # Strictly enforce Q_QUERY size to prevent evaluation size bias
    # (assumes the test split is large enough to handle 1000 per class)
    q_size = min(Config.Q_QUERY, len(remaining_mal), len(remaining_ben))

    idx_mal_q = np.random.choice(remaining_mal, q_size, replace=False)
    idx_ben_q = np.random.choice(remaining_ben, q_size, replace=False)

    # Batches
    X_sup = torch.cat([X_mal[idx_mal_s], X_ben[idx_ben_s]])
    y_sup = torch.cat([torch.ones(k_shot), torch.zeros(k_shot)]).long()

    X_qry = torch.cat([X_mal[idx_mal_q], X_ben[idx_ben_q]])
    y_qry = torch.cat([torch.ones(q_size), torch.zeros(q_size)]).long()

    # Extract Features
    with torch.no_grad():
        z_sup, _ = ae_model(X_sup.to(Config.DEVICE))
        z_qry, _ = ae_model(X_qry.to(Config.DEVICE))

    return z_sup, y_sup.to(Config.DEVICE), z_qry, y_qry.to(Config.DEVICE)


def manual_maml_step(model, z_sup, y_sup, z_qry, y_qry, train=True):
    fast_weights = dict(model.named_parameters())

    # Inner Loop (Task Adaptation)
    for _ in range(Config.INNER_STEPS):
        out = torch.func.functional_call(model, fast_weights, z_sup)
        loss = nn.CrossEntropyLoss()(out, y_sup)
        grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=train)
        fast_weights = {n: p - Config.INNER_LR * g for (n, p), g in zip(fast_weights.items(), grads)}

    # Outer Loop (Meta-Optimization)
    out_q = torch.func.functional_call(model, fast_weights, z_qry)

    if train:
        meta_loss = nn.CrossEntropyLoss()(out_q, y_qry)
        return meta_loss
    else:
        preds = torch.argmax(out_q, dim=1)
        return preds, y_qry


def calculate_metrics(y_true, y_pred):
    """Compute evaluation metrics from Table III of the paper."""
    y_true_np = y_true.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    return (accuracy_score(y_true_np, y_pred_np),
            f1_score(y_true_np, y_pred_np, average='binary', zero_division=0),
            precision_score(y_true_np, y_pred_np, average='binary', zero_division=0),
            recall_score(y_true_np, y_pred_np, average='binary', zero_division=0),
            matthews_corrcoef(y_true_np, y_pred_np))


def print_confusion_matrix(y_true_all, y_pred_all, k):
    """Print a confusion matrix for a given K-shot setting (Fig. 5)."""
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"   Confusion Matrix (K={k}-shot):")
    print(f"                  Predicted")
    print(f"                  Benign   Malware")
    print(f"   Actual Benign  {tn:<8} {fp}")
    print(f"   Actual Malware {fn:<8} {tp}")
    print(f"   (TN={tn}, FP={fp}, FN={fn}, TP={tp})")


if __name__ == "__main__":
    # Set seeds for reproducibility
    set_seed(Config.SEED)

    print("="*70)
    print("AFE-MAML: Few-Shot Malware Detection with Autoencoders + MAML")
    print("="*70)
    print(f"Device: {Config.DEVICE}")
    print(f"Hyperparameters: alpha={Config.INNER_LR}, beta={Config.META_LR}, "
          f"inner_steps={Config.INNER_STEPS}")
    print()

    # Preprocess Full Data (Section II-A)
    X_np, y_np, label_encoder = preprocess_dataset(Config.DATASET_PATH)

    # Validate feature dimension matches CONFIG
    assert X_np.shape[1] == Config.INPUT_DIM, (
        f"Feature count {X_np.shape[1]} != expected INPUT_DIM {Config.INPUT_DIM}. "
        f"Check preprocessing or update Config.INPUT_DIM."
    )

    y_tensor = torch.LongTensor(y_np)

    # ----------------------------------------------------------------------
    # Stratified train/test split before scaling to prevent data leakage
    # ----------------------------------------------------------------------
    X_train_np, X_test_np, y_train, y_test = train_test_split(
        X_np, y_tensor, test_size=0.25, stratify=y_tensor, random_state=42
    )

    # Min-Max normalization (Eq. 1): fit on training data only, transform both
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_np)
    X_test_scaled = scaler.transform(X_test_np)
    print(f"  Scaler fit on training split ({X_train_np.shape[0]} samples), "
          f"applied to test split ({X_test_np.shape[0]} samples)")

    X_train = torch.FloatTensor(X_train_scaled)
    X_test = torch.FloatTensor(X_test_scaled)

    # Create subsets for Meta-Training
    X_mal_train = X_train[y_train == 1]
    X_ben_train = X_train[y_train == 0]

    # Create subsets for Meta-Testing
    X_mal_test = X_test[y_test == 1]
    X_ben_test = X_test[y_test == 0]


    # ------------------------------------------------------------------
    # Stage 1: Train Autoencoder exclusively on Training Split (Eq. 3)
    # Architecture: 72 -> 256 -> 128 -> 64 -> 128 -> 256 -> 72
    # Loss: MSE reconstruction loss
    # ------------------------------------------------------------------
    print(f"\n[Stage 1] Training Autoencoder for {Config.AE_EPOCHS} epochs "
          f"(lr={Config.AE_LR}, batch_size={Config.AE_BATCH_SIZE})")
    ae = Autoencoder().to(Config.DEVICE)
    opt_ae = optim.Adam(ae.parameters(), lr=Config.AE_LR)
    mse_criterion = nn.MSELoss()

    loader = DataLoader(TensorDataset(X_train), batch_size=Config.AE_BATCH_SIZE,
                        shuffle=True)

    ae.train()
    for epoch in range(Config.AE_EPOCHS):
        epoch_loss = 0.0
        n_batches = 0
        for (batch_x,) in loader:
            batch_x = batch_x.to(Config.DEVICE)
            _, recon = ae(batch_x)
            loss = mse_criterion(recon, batch_x)
            opt_ae.zero_grad()
            loss.backward()
            opt_ae.step()
            epoch_loss += loss.item()
            n_batches += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            avg_loss = epoch_loss / n_batches
            print(f"   Epoch [{epoch+1:3d}/{Config.AE_EPOCHS}]  "
                  f"Reconstruction Loss: {avg_loss:.6f}")
    ae.eval()
    print("Autoencoder training complete.\n")

    k_values = [1, 2, 5, 10, 20, 50]
    results_summary = []

    print(f"[Stage 2] Meta-Learning Evaluation for K = {k_values}")
    print(f"   Meta-training episodes: {Config.META_EPISODES}, "
          f"Meta-testing episodes: {Config.TEST_EPISODES}")
    print(f"   Query set: {Config.Q_QUERY} per class ({Config.Q_QUERY * 2} total)")

    for k in k_values:
        print(f"\n{'='*50}")
        print(f"Running K={k}-Shot (2-way {k}-shot)")
        print(f"{'='*50}")
        Config.K_SHOT = k

        # Re-initialize MAML classifier for each K-shot setting
        maml = MAMLClassifier().to(Config.DEVICE)

        # Outer-loop optimizer: Adam with meta-learning rate beta (Eq. 6)
        meta_optimizer = optim.Adam(maml.parameters(), lr=Config.META_LR)
        ce_criterion = nn.CrossEntropyLoss()

        # -- Meta-Training Loop (uses training split only) --
        maml.train()
        start_time = time.time()
        running_loss = 0.0
        for episode in range(Config.META_EPISODES):
            # Sample few-shot episode from training data
            z_s, y_s, z_q, y_q = get_few_shot_batch(
                X_mal_train, X_ben_train, ae, k)

            meta_optimizer.zero_grad()
            loss = manual_maml_step(maml, z_s, y_s, z_q, y_q, train=True)
            loss.backward()
            meta_optimizer.step()

            running_loss += loss.item()
            if (episode + 1) % 200 == 0:
                avg = running_loss / 200
                print(f"   Episode [{episode+1:4d}/{Config.META_EPISODES}]  "
                      f"Meta-Loss: {avg:.4f}")
                running_loss = 0.0

        train_time = time.time() - start_time
        print(f"   Training complete in {train_time:.1f}s")

        # -- Meta-Testing Loop (uses held-out test split) --
        maml.eval()
        metrics_accum = {"Acc": [], "F1": [], "Prec": [], "Rec": [], "MCC": []}
        all_preds = []
        all_targets = []

        for _ in range(Config.TEST_EPISODES):
            z_s, y_s, z_q, y_q = get_few_shot_batch(
                X_mal_test, X_ben_test, ae, k)
            preds, targets = manual_maml_step(
                maml, z_s, y_s, z_q, y_q, train=False)

            acc, f1, prec, rec, mcc = calculate_metrics(targets, preds)
            metrics_accum["Acc"].append(acc)
            metrics_accum["F1"].append(f1)
            metrics_accum["Prec"].append(prec)
            metrics_accum["Rec"].append(rec)
            metrics_accum["MCC"].append(mcc)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

        # Aggregate confusion matrix across all test episodes (Fig. 5)
        all_preds_np = np.concatenate(all_preds)
        all_targets_np = np.concatenate(all_targets)
        print_confusion_matrix(all_targets_np, all_preds_np, k)

        res = {
            "K-Shot": k,
            "Accuracy": np.mean(metrics_accum["Acc"]),
            "F1-Score": np.mean(metrics_accum["F1"]),
            "Precision": np.mean(metrics_accum["Prec"]),
            "Recall": np.mean(metrics_accum["Rec"]),
            "MCC": np.mean(metrics_accum["MCC"]),
            "Acc_std": np.std(metrics_accum["Acc"]),
            "F1_std": np.std(metrics_accum["F1"]),
        }
        results_summary.append(res)
        print(f"   Avg Metrics ({Config.TEST_EPISODES} episodes):")
        print(f"     Accuracy:  {res['Accuracy']:.4f} +/- {res['Acc_std']:.4f}")
        print(f"     F1-Score:  {res['F1-Score']:.4f} +/- {res['F1_std']:.4f}")
        print(f"     Precision: {res['Precision']:.4f}")
        print(f"     Recall:    {res['Recall']:.4f}")
        print(f"     MCC:       {res['MCC']:.4f}")

    # -- Final Summary Table (Table V) --
    print(f"\n{'='*70}")
    print("Performance Across Different Few-Shot Settings (Table V)")
    print(f"{'='*70}")
    header = (f"{'K-Shot':>8} | {'Accuracy':>10} | {'F1-Score':>10} | "
              f"{'Precision':>10} | {'Recall':>10} | {'MCC':>10}")
    print(header)
    print("-" * len(header))
    for res in results_summary:
        print(f"{res['K-Shot']:>8} | {res['Accuracy']:>10.4f} | "
              f"{res['F1-Score']:>10.4f} | {res['Precision']:>10.4f} | "
              f"{res['Recall']:>10.4f} | {res['MCC']:>10.4f}")

NameError: name '__file__' is not defined